In [ ]:
# Cell 1: Setup
"""
# ArcticDEM Elevation History Analysis
Extract and plot elevation time series at a point using ArcticDEM strip data.
"""

# Add the parent directory to path if needed
import sys
sys.path.append('..')
import importlib
import os
from config import OUTPUT_DIR, ARCHIVE_DIR
import analysis
importlib.reload(analysis)
from analysis import run_elevation_history, batch_process_elevation_histories_coregistered, batch_process_elevation_histories_gz, plot_elevation_history
from elevation_utils import search_arcticdem_strips, filter_strip_dems, get_dem_metadata, get_strip_ids, parse_coordinate_string
from interactive_maps import select_point_interactive
import numpy as np
import geopandas as gpd
import time
from ipyleaflet import (
    AwesomeIcon,
    FullScreenControl,
    Map,
    Marker,
    basemaps,
)
from IPython.display import clear_output, display
from ipywidgets import HTML, Button, HBox, Layout, VBox, widgets  # type: ignore


# STAC API needs to be defined in this file because it uses global coords (I think)
# STAC API needs to be defined in this file because it uses global coords (I think)
def stac_api(timeout=2):
    """
    Interactive map to select a single point with draggable marker.
    
    Displays a leaflet map where users can drag a marker and click
    'Confirm' to select coordinates.
    
    Parameters
    ----------
    timeout : int
        Maximum seconds to wait for user confirmation
        
    Returns
    -------
    tuple
        (longitude, latitude) in WGS84
    """
    global coords
    coords = (-54.2, 75)
    mid_lat = 75.0
    mid_lon = -54.2

    # Create a leaflet map widget to find an AOI for the spatial query of the STAC API
    m = Map(
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
        center=(mid_lat, mid_lon),
        zoom=6,
        layout=Layout(height="380px", width="700px"),
    )
    m.add_control(FullScreenControl())
    result_output = widgets.Output()

    # Create info display
    info_html = HTML(
        value="<b>Drag the marker, then click Confirm</b>",
        layout=Layout(padding="10px"),
    )

    # Create confirmation button
    confirm_button = Button(
        description="Confirm Location",
        button_style="success",
        disabled=False,
        tooltip="Click after placing marker",
    )

    # Create markers (initially slightly offset from center)
    centre_marker = Marker(
        # location=(center_y, center_x - 0.1),
        location=(mid_lat, mid_lon),  # Leaflet expects (lat, lon)
        draggable=True,
        name="Location",
        icon=AwesomeIcon(name="map-pin", marker_color="red"),
    )
    m.add_layer(centre_marker)

    # Update info display when marker moves
    def update_display(*args):
        # Remember: Leaflet uses (lat, lon) format, but we want to store as (lon, lat)
        centre_lat, centre_lon = centre_marker.location[0], centre_marker.location[1]
        info_html.value = (
            f"<b>Current Position:</b><br>"
            f"<span style='color:red'>Location:</span> Lon: {centre_lon:.3f}, Lat: {centre_lat:.3f}<br>"
        )

    # Handle confirmation
    def on_confirm(b):
        global coords
        # Store as (lon, lat) for consistency with GIS conventions
        centre_lat, centre_lon = centre_marker.location[0], centre_marker.location[1]
        with result_output:
            result_output.clear_output()
            print(f"Centre confirmed! (lon, lat): {centre_lon}, {centre_lat}")
            global coords
            coords = (centre_lon, centre_lat)

    # Connect callbacks
    confirm_button.on_click(on_confirm)
    centre_marker.observe(update_display, names=["location"])
    update_display()  # Initial display update

    # Display all components
    display(VBox([info_html, m, HBox([confirm_button]), result_output]))

    # Wait for confirmation with timeout
    start_time = time.time()
    while (time.time() - start_time) < timeout:
        time.sleep(0.1)

    clear_output(wait=True)  # Clean up the display
    return coords

print("✓ Modules loaded")

In [ ]:
# Cell 2: Configure Parameters
"""
## Configuration
Edit these parameters for your analysis.a
"""
# Analysis parameters
TIME_RANGE = "2009-01-01/2026-12-31"  # YYYY-MM-DD/YYYY-MM-DD
WINDOW_SIZE = 5                        # side size of the window in pixels (1 = single pixel, 3 = 3x3 window, etc.)
WINDOW_TYPE = 'square'                 # 'square' or 'cross'
COREG_MODE = 'mosaic'                    # 'none' (faster, as it gets the elevation from compressed data), 'altim', or 'mosaic'
MAX_CLOUD_COVER = 0.99                  # Maximum cloud cover fraction for filtering DEMs
LAKE_NAME = 'lakes'                       # Optional name for the lake or area of interest (used in plot titles)

# ============================================================================
# SELECT INPUT METHOD
# ============================================================================

INPUT_METHOD = 1  # <-- CHANGE THIS: 1=Interactive, 2=Manual, 3=GeoPackage batch

print(f"Input method: {INPUT_METHOD}")
print(f"  1 = Interactive map")
print(f"  2 = Manual coordinate entry")
print(f"  3 = Batch from GeoPackage/GeoParquet file")

###################################################
print(f"Archive: {ARCHIVE_DIR}")
print(f"Time range: {TIME_RANGE}")
print(f"Window: {WINDOW_SIZE}x{WINDOW_SIZE} {WINDOW_TYPE}")
print(f"Coregistration: {COREG_MODE}")
print(f"Max cloud cover: {MAX_CLOUD_COVER}")


In [ ]:
# Cell 3: Select Coordinates
"""
## Select Location
Choose how to define your point(s) for elevation history extraction:
  1. Interactive map (click and drag marker)
  2. Type coordinates manually
  3. Batch from GeoPackage/GeoParquet file (multiple points)
"""
# ============================================================================
# METHOD 1: Interactive Map
# ============================================================================
if INPUT_METHOD == 1:
    print("\nUse the map to select your location...")
    print("Drag the RED marker to your point of interest, then click Confirm")
    
    coords = stac_api(
        timeout=2
    )
    
# ============================================================================
# METHOD 2: Manual Coordinates
# ============================================================================
elif INPUT_METHOD == 2:
    coord_input = input(
        "Enter coordinates as lon,lat (e.g. -48.30,63.85): "
    ).strip()
    
    # Use default if empty
    if not coord_input:
        coord_input = "-48.30,63.85"  # <-- SET YOUR DEFAULT HERE
        print(f"Using default coordinates: {coord_input}")
    
    coords = parse_coordinate_string(coord_input)
    
    point_list = [{'coords': coords, 'label': 'Point_1', 'index': 0}]
    print(f"\n✓ Coordinates: ({coords[0]:.6f}°E, {coords[1]:.6f}°N)")

# ============================================================================
# METHOD 3: Batch from GeoPackage/GeoParquet
# ============================================================================
elif INPUT_METHOD == 3:  
    # Path to GeoPackage or GeoParquet file
    file_path = input(
        "Enter path to GeoPackage (.gpkg) or GeoParquet (.parquet) file: "
    ).strip()
    
    # Use default if empty
    if not file_path:
        file_path = "/home/moralpom/luna/CPOM/moralpom/globe/data/ArcticDEM/sample_diego_sites_for_notebook.gpkg"  # <-- SET DEFAULT
        print(f"Using default file: {file_path}")
    
    # Determine file type
    if file_path.endswith('.parquet'):
        gdf = gpd.read_parquet(file_path)
    elif file_path.endswith('.gpkg'):
        gdf = gpd.read_file(file_path)
    else:
        # Try both
        try:
            gdf = gpd.read_file(file_path)
        except:
            gdf = gpd.read_parquet(file_path)
    
    print(f"\nLoaded {len(gdf)} features from: {file_path}")
    print(f"  Columns: {list(gdf.columns)}")
    print(f"  Geometry type: {gdf.geometry.geom_type.unique()}")
    
    # Extract coordinates from geometry
    if gdf.geometry is not None and not gdf.geometry.is_empty.all():
        # Get coordinates from geometry (works for Points)
        # For Points, extract directly
        if all(gdf.geometry.geom_type == 'Point'):
            lons = gdf.geometry.x.values
            lats = gdf.geometry.y.values
        else:
            # For other geometry types, use centroid
            print("  Using centroids of geometries...")
            centroids = gdf.geometry.centroid
            lons = centroids.x.values
            lats = centroids.y.values
    else:
        raise ValueError("No valid geometry found in file")
    
    # Try to find a label/name column
    label_col = None
    for col in ['name', 'Name', 'NAME', 'label', 'Label', 'id', 'ID', 'lake_name', 'site']:
        if col in gdf.columns:
            label_col = col
            break
    
    if label_col:
        labels = gdf[label_col].values
    else:
        labels = [f"Point_{i+1}" for i in range(len(gdf))]
    
    # Build point list
    point_list = []
    for i, (lon, lat, label) in enumerate(zip(lons, lats, labels)):
        # Handle NaN coordinates
        if np.isnan(lon) or np.isnan(lat):
            print(f"  ⚠️ Skipping {label}: invalid coordinates")
            continue
        
        point_list.append({
            'coords': (float(lon), float(lat)),
            'label': str(label),
            'index': i,
        })
        
        print(f"  {i+1}. {label}: ({lon:.6f}°E, {lat:.6f}°N)")
    
    print(f"\n  ✓ Extracted {len(point_list)} valid points")


In [ ]:
# Cell 4: Run Analysis for All Points
"""
## Process Elevation History
Query STAC API and extract elevations from all available DEMs
for each selected point.
"""

import time
from datetime import datetime


if INPUT_METHOD == 1 or INPUT_METHOD == 2:
    # Wrap in list for consistent format
    point_list = [{'coords': coords, 'label': 'Point_1', 'index': 0}]
    print(f"\n✓ Selected coordinates: ({coords[0]:.6f}°E, {coords[1]:.6f}°N)")

# ============================================================================
# DISPLAY ALL POINTS
# ============================================================================
print(f"\n{'='*60}")
print(f"POINTS TO PROCESS: {len(point_list)}")
print(f"{'='*60}")
# point_list = point_list[23:]
for i, pt in enumerate(point_list):
    print(f"  {i+1}. [{pt['label']}] ({pt['coords'][0]:.6f}°E, {pt['coords'][1]:.6f}°N)")

# Store the list for use in the next cell
print(f"\n✓ {len(point_list)} point(s) ready for processing")

# Store results for all points
all_history_results = []

# Keep track of total time
start_time = time.time()

for i, pt in enumerate(point_list):
    coords = pt['coords']
    label = pt['label']
    
    print(f"\n{'='*60}")
    print(f"PROCESSING POINT {i+1}/{len(point_list)}: {label}")
    print(f"  Coordinates: ({coords[0]:.6f}°E, {coords[1]:.6f}°N)")
    print(f"{'='*60}")
    
    # STAC search for this point
    bbox = (
        coords[0] - 0.001,  # West
        coords[1] - 0.001,  # South
        coords[0] + 0.001,  # East
        coords[1] + 0.001,  # North
    )
    
    items_gdf, items = search_arcticdem_strips(bbox, TIME_RANGE)
    
    if len(items_gdf) == 0:
        print(f"  ⚠️ No DEMs found for {label} - skipping")
        all_history_results.append({
            'label': label,
            'coords': coords,
            'error': 'No DEMs found',
            'history': None,
        })
        continue
    
    items_gdf = filter_strip_dems(items_gdf, max_cloud_cover=0.2)
    pairnames, geocells, dates = get_dem_metadata(items_gdf)
    strip_ids = get_strip_ids(items_gdf)  # pins the strip segment
    
    print(f"  Found {len(pairnames)} DEMs for this point")
    
    # Process elevation history
    try:
        if COREG_MODE == 'none':
            history = batch_process_elevation_histories_gz(
                geocells=geocells,
                pairnames=pairnames,
                dates=dates,
                coords=coords,
                archdir=ARCHIVE_DIR,
                output_path=OUTPUT_DIR,
                window_size=WINDOW_SIZE,
                window_type=WINDOW_TYPE,
                time_range=TIME_RANGE,
                strip_ids=strip_ids,
            )
        else:
            history = batch_process_elevation_histories_coregistered(
                geocells=geocells,
                pairnames=pairnames,
                dates=dates,
                coords=coords,
                archdir=ARCHIVE_DIR,
                output_path=OUTPUT_DIR,
                coreg_mode=COREG_MODE,
                window_size=WINDOW_SIZE,
                window_type=WINDOW_TYPE,
                time_range=TIME_RANGE,
                strip_ids=strip_ids,
            )
        
        # Generate plot
        plot_path = plot_elevation_history(history, OUTPUT_DIR, COREG_MODE)
        
        # Save data file
        suf = {'none': '_nc', 'altim': '_altim', 'mosaic': '_mosaic'}.get(COREG_MODE, '_nc')
        
        # Years from the input selection (TIME_RANGE), same as the plot filename
        year_i, year_f = history['year_span']
        years_str = f"{year_i}-{year_f}"
        
        # Clean label for filename
        safe_label = label.replace(' ', '_').replace('/', '_')[:30]
        
        data_path = os.path.join(
            OUTPUT_DIR, "elevation_histories",
            f"elevation_history_{safe_label}_{coords[0]:.3f}_{coords[1]:.3f}{suf}_{years_str}.txt"
        )
        os.makedirs(os.path.dirname(data_path), exist_ok=True)
        
        with open(data_path, 'w') as f:
            f.write(f"Elevation History for {label}\n")
            f.write(f"Coordinates: ({coords[0]:.6f}°E, {coords[1]:.6f}°N)\n")
            f.write(f"Coregistration: {COREG_MODE}\n")
            f.write(f"Window: {WINDOW_SIZE}x{WINDOW_SIZE} {WINDOW_TYPE}\n")
            f.write("Date, Elevation (m), Std (m), Valid Pixels, Pairname\n")
            for date, elev, std, vp, pair in zip(
                history['dates'], history['elevations'],
                history['elevations_std'], history['valid_pixels'],
                history['pairnames']
            ):
                f.write(f"{date.isoformat()}, {elev:.2f}, {std:.2f}, {vp}, {pair}\n")
        
        # Store results
        valid_count = sum(1 for m in history.get('metadata', []) if m.get('valid', False))
        
        all_history_results.append({
            'label': label,
            'coords': coords,
            'history': history,
            'plot_path': plot_path,
            'data_path': data_path,
            'n_dems': len(pairnames),
            'n_valid': valid_count,
            'error': None,
        })
        
        print(f"  ✓ {label}: {valid_count}/{len(pairnames)} valid elevations")
        print(f"  Plot: {os.path.basename(plot_path) if plot_path else 'N/A'}")
        
    except Exception as e:
        print(f"  ✗ Error processing {label}: {e}")
        import traceback
        traceback.print_exc()
        
        all_history_results.append({
            'label': label,
            'coords': coords,
            'error': str(e),
            'history': None,
        })

# Print summary
elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"Total points: {len(point_list)}")
print(f"Successful: {sum(1 for r in all_history_results if r.get('history') is not None)}")
print(f"Failed: {sum(1 for r in all_history_results if r.get('error'))}")
print(f"Time elapsed: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")

# Quick summary table
print(f"\n{'Label':<30} {'DEMs':>6} {'Valid':>6} {'Status':>10}")
print("-" * 55)
for r in all_history_results:
    if r.get('history'):
        print(f"{r['label']:<30} {r['n_dems']:>6} {r['n_valid']:>6} {'✓ OK':>10}")
    else:
        print(f"{r['label']:<30} {'-':>6} {'-':>6} {'✗ FAIL':>10}")

In [ ]:
# Cell 5: View Results
"""
## Results
Display plots and statistics for all processed points.
"""

from IPython.display import Image, display
import numpy as np

# First, show summary
print(f"\n{'='*60}")
print(f"RESULTS SUMMARY")
print(f"{'='*60}")

for i, result in enumerate(all_history_results):
    label = result['label']
    coords = result['coords']
    
    print(f"\n{'─'*60}")
    print(f"Point {i+1}: {label}")
    print(f"  Coordinates: ({coords[0]:.6f}°E, {coords[1]:.6f}°N)")
    
    if result.get('error'):
        print(f"  ❌ Error: {result['error']}")
        continue
    
    history = result.get('history')
    if history is None:
        print(f"  ❌ No data")
        continue
    
    # Statistics
    elevations = np.array(history['elevations'])
    valid = elevations[~np.isnan(elevations)]
    
    if len(valid) > 0:
        # Calculate trend
        dates = history['dates']
        valid_dates = [d for d, e in zip(dates, elevations) if not np.isnan(e)]
        
        if len(valid) >= 3 and len(valid_dates) >= 3:
            from scipy import stats
            years_num = np.array([d.year + d.timetuple().tm_yday / 365.25 
                                 for d in valid_dates])
            slope, intercept, r_value, p_value, std_err = stats.linregress(
                years_num[:len(valid)], valid[:len(valid)]
            )
            
            print(f"  Valid elevations: {len(valid)}/{len(elevations)}")
            print(f"  Mean elevation: {np.mean(valid):.1f} ± {np.std(valid):.1f} m")
            print(f"  Min/Max: {np.min(valid):.1f} / {np.max(valid):.1f} m")
            print(f"  Trend: {slope:+.3f} m/yr (R²={r_value**2:.3f}, p={p_value:.4f})")
            
            if slope < -0.1:
                print(f"  ⚠️ Significant thinning detected!")
            elif slope < -0.02:
                print(f"  📉 Moderate thinning")
            elif slope > 0.02:
                print(f"  📈 Thickening")
            else:
                print(f"  ➡️ Stable")
        else:
            print(f"  Valid elevations: {len(valid)}/{len(elevations)}")
            print(f"  Mean elevation: {np.mean(valid):.1f} ± {np.std(valid):.1f} m")
    else:
        print(f"  No valid elevation data")

# Then show first few plots
print(f"\n{'='*60}")
print(f"ELEVATION HISTORY PLOTS")
print(f"{'='*60}")

# Show up to 5 plots
n_show = min(5, len(all_history_results))

for i in range(n_show):
    result = all_history_results[i]
    
    if result.get('plot_path') and os.path.exists(result['plot_path']):
        print(f"\n{result['label']}:")
        display(Image(filename=result['plot_path']))
    elif result.get('error'):
        print(f"\n{result['label']}: ❌ {result['error']}")

if len(all_history_results) > n_show:
    print(f"\n... and {len(all_history_results) - n_show} more points")
    print(f"Check the output directory for all plots:")
    print(f"  {os.path.join(OUTPUT_DIR, 'elevation_histories')}")

In [ ]:
# Cell 5: OPTIONAL
'''
## Batch history processing
Batch Processing of Multiple Points from a GeoParquet File
'''
import geopandas as gpd
from shapely import get_coordinates
from tqdm import tqdm

# Replace the string below with the actual GeoParquet file path.
geoparquet_path = path = '/home/moralpom/luna/CPOM/moralpom/globe/data/lakes_sites/Sub_GL_diego_sites_updated.parquet' 
gdf = gpd.read_parquet(geoparquet_path)

if "geometry" not in gdf.columns:
	raise ValueError("The GeoParquet file does not contain a 'geometry' column.")

if "Name" not in gdf.columns:
	raise ValueError("The GeoParquet file does not contain a 'Name' column.")

# Extract all coordinate pairs from the geometries and their names associated
coords_total = [tuple(map(float, pt)) for pt in get_coordinates(gdf.geometry).tolist()]
coords_total = list(dict.fromkeys(coords_total))  # Optional: remove duplicates while preserving order
names = [name for name in gdf.Name.tolist()]
# Edit names so it substitutes spaces with underscores and removes special characters for file naming
names = [name.replace(" ", "_").replace("/", "_").replace("\\", "_") for name in names]

print(f"Loaded {len(gdf)} features from: {geoparquet_path}")
print(f"Extracted {len(coords_total)} coordinate pairs")

for i in tqdm(range(len(coords_total)), desc="Processing pairs of coordinates"):
    coords_i = coords_total[i]
    LAKE_NAME = names[i]
    
    history = run_elevation_history(
        archdir=ARCHIVE_DIR,
        coords=coords_i,
        time_range=TIME_RANGE,
        window_size=WINDOW_SIZE,
        window_type=WINDOW_TYPE,
        coreg_mode=COREG_MODE,
        max_cloud_cover=MAX_CLOUD_COVER,
        lake_name=LAKE_NAME
    )

    print("\n✓ Analysis complete!")
    print(f"Processed {len(history['dates'])} DEMs")
    print(f"Valid elevations: {sum(1 for m in history['metadata'] if m.get('valid', False))}")